# BMLL | Corrected four-year portfolio and order book (research replay)

**Analysis window:** 2022-07-01 to 2026-06-30. **Bootstrap:** June 2022 post-rebalance decision.

This revision fixes the earlier four-year *global exclusion* bug: a single unresolved event or temporary price issue must not exclude a security for the entire history. It also uses WRDS raw USD dividends as a **provisional research-only convention** when the Bloomberg discrepancy exactly matches a later CRSP-verified split factor or when source economics are otherwise identical.

**Certification is NOT automatic.** Unapproved events, ticker-only matches, complex actions, vendor data gaps, and missing benchmark/identifier evidence remain flagged. Real WRDS access is required; the result must be independently validated after the live run.

No Python source or vendor data is embedded in this notebook.

## 0. Configuration

Change dates and source release here, before running any subsequent cell. Defaults represent four complete analysis years.

In [ ]:
ANALYSIS_START = '2022-07-01'
END_DATE = '2026-06-30'
REBALANCE_START = '2022-01-01'  # discover the June-2022 starting holdings
CRSP_SCHEMA = 'crsp_q_stock'   # try crsp_a_stock if the quarterly release lacks history
AUM = 10_000_000_000.0
CASH_BUFFER_BPS = 10.0
FEE_BPS = 0.0
RUN_OFFLINE_TESTS = True
RUN_WRDS_EXTRACTION = True
RUN_REPLAY = True

from datetime import date
assert ANALYSIS_START == '2022-07-01' and END_DATE == '2026-06-30' or ANALYSIS_START < END_DATE
print(f'Analysis window: {ANALYSIS_START} through {END_DATE} | AUM = ${AUM:,.0f}')
# RESEARCH ONLY: deterministic WRDS convention for narrow documented exceptions.
# This does not make corporate actions human-approved or certify full-index coverage.
PROVISIONAL_WRDS_RESEARCH = True


## 1. Source files and Bloomberg upload

When running **from the GitHub checkout root**, the notebook automatically uses repository Python modules and asks only for the Bloomberg workbook.
When running **standalone in Colab**, upload the `BMLL_4Y_Coverage_Fix_Support.zip` and Bloomberg workbook together.
Raw Bloomberg/WRDS data must remain outside version control.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, uuid
from datetime import datetime, timezone

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

# A git checkout already contains the source modules; no source ZIP needed.
_in_repo = all((Path.cwd()/name).is_file() for name in [
    'research_pipeline.py',
    'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb',
    'scripts/coverage_recovery.py',
    'scripts/complete_release.py',
    'scripts/build_wrds_handoff.py',
    'scripts/extract_bloomberg.py',
])
if _in_repo:
    PROJECT_DIR = Path.cwd().resolve()
    RUN_ROOT = PROJECT_DIR
    if colab_files is not None:
        print('Repository files detected. Upload Bloomberg Excel workbook only (plus optional review CSVs).')
        _uploads = colab_files.upload()
        workbooks = [Path(k).resolve() for k in _uploads if k.lower().endswith('.xlsx')]
    else:
        workbooks = [Path(input('Bloomberg .xlsx workbook path: ').strip()).expanduser().resolve()]
    if len(workbooks) != 1 or not workbooks[0].is_file():
        raise ValueError('Specify exactly one Bloomberg workbook.')
    PRIVATE_DIR = PROJECT_DIR/'data/private_uploads'
    PRIVATE_DIR.mkdir(parents=True, exist_ok=True)
    BLOOMBERG_XLSX = PRIVATE_DIR/workbooks[0].name
    if workbooks[0] != BLOOMBERG_XLSX:
        shutil.copy2(workbooks[0], BLOOMBERG_XLSX)
    sys.path.insert(0,str(PROJECT_DIR))
    print('Using checked-out source at', PROJECT_DIR)
else:
    if colab_files is not None:
        print('Upload BMLL_4Y_Coverage_Fix_Support.zip AND BMLL_Bloomberg - FINAL.xlsx together.')
        uploaded = colab_files.upload()
        items = [Path(k).resolve() for k in uploaded]
    else:
        print('Local Jupyter: enter paths to the ZIP and Bloomberg XLSX.')
        items = [Path(input('Support ZIP: ').strip()).expanduser().resolve(),
                 Path(input('Bloomberg workbook: ').strip()).expanduser().resolve()]
        if not all(p.is_file() for p in items):
            raise FileNotFoundError('At least one input path does not exist')

    source_zips = []
    for path in items:
        if path.suffix.lower() == '.zip':
            with zipfile.ZipFile(path) as z:
                names = z.namelist()
                if any(x.endswith('scripts/coverage_recovery.py') for x in names) and \
                   any(x.endswith('research_pipeline.py') for x in names):
                    source_zips.append(path)
    if len(source_zips) != 1:
        raise ValueError('Upload exactly one BMLL_4Y_Coverage_Fix_Support.zip (other ZIPs lack coverage recovery).')
    workbooks = [p for p in items if p.suffix.lower() == '.xlsx']
    if len(workbooks) != 1:
        raise ValueError('Upload exactly one Bloomberg .xlsx workbook')

    RUN_ROOT = Path('/content' if colab_files is not None else Path.cwd()) / ('BMLL_4Y_'+uuid.uuid4().hex[:8])
    RUN_ROOT.mkdir(parents=True,exist_ok=False)
    with zipfile.ZipFile(source_zips[0]) as archive:
        for info in archive.infolist():
            p=Path(info.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Untrusted ZIP contains an unsafe path')
        archive.extractall(RUN_ROOT)
    PROJECT_DIR = RUN_ROOT/'BMLL_Notebook_Release_Kit'
    assert (PROJECT_DIR/'scripts/coverage_recovery.py').is_file()
    assert (PROJECT_DIR/'scripts/multiyear_preflight.py').is_file()
    assert (PROJECT_DIR/'scripts/build_wrds_handoff.py').is_file()
    assert (PROJECT_DIR/'scripts/complete_release.py').is_file()
    assert (PROJECT_DIR/'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb').is_file()
    PRIVATE_DIR = PROJECT_DIR/'data/private_uploads'
    PRIVATE_DIR.mkdir(parents=True,exist_ok=True)
    BLOOMBERG_XLSX = PRIVATE_DIR / workbooks[0].name
    shutil.copy2(workbooks[0],BLOOMBERG_XLSX)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0,str(PROJECT_DIR))
    print('Project:',PROJECT_DIR)
    print('Bloomberg:',BLOOMBERG_XLSX,'| MB:',round(BLOOMBERG_XLSX.stat().st_size/1e6,2))

## 2. Install dependencies and run checks

Requires WRDS Python package, `psycopg2`, Pandas and OpenPyXL. No user credentials are stored in the notebook.

In [ ]:
import subprocess

command=[sys.executable,'-m','pip','install','-q','-r',str(PROJECT_DIR/'requirements.txt'),
         'psycopg2-binary>=2.9','openpyxl>=3.1','nbformat>=5']
subprocess.check_call(command)
if RUN_OFFLINE_TESTS:
    t=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
        cwd=PROJECT_DIR,text=True,capture_output=True)
    print((t.stdout+t.stderr)[-6500:])
    if t.returncode:
        raise RuntimeError('Offline regression tests failed. Do not begin a four-year run.')

## 3. Normalize Bloomberg corporate actions

Converts only the workbook you upload. QA flags and blank PERMNOs are retained for historical WRDS identifier matching, not assigned speculative ticker matches.

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.extract_bloomberg import convert

BLOOMBERG_CSV = PROJECT_DIR/'data/BMLL_Bloomberg_Corporate_Actions.csv'
BLOOMBERG_QC = PROJECT_DIR/'data/bloomberg_review_flags.csv'
BLOOMBERG_REPORT = PROJECT_DIR/'data/bloomberg_conversion_report.json'
convert(BLOOMBERG_XLSX,BLOOMBERG_CSV,BLOOMBERG_QC,BLOOMBERG_REPORT)
print(json.dumps(json.loads(BLOOMBERG_REPORT.read_text())['counts'],indent=2))
print('Bloomberg export ready; raw workbook remains private.')

## 4. Authenticate WRDS and discover the bootstrap date

The fund's first **decision snapshot** must precede July 2022, because a July 1 initialization would trade with weights calculated after the earlier June rebalance. The date is discovered from your actual CRSP release, not assumed to be a trading day.

In Colab, create `WRDS_USERNAME` and `WRDS_PASSWORD` Secrets and enable notebook access, or use the secure prompts.

In [ ]:
from research_pipeline import load_helpers
helpers = load_helpers(PROJECT_DIR/'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb')
if RUN_WRDS_EXTRACTION:
    with helpers['wrds_session']() as db:
        bootstrap_weights, bootstrap_diagnostics = helpers['sp500_cap_weights_quarterly'](
            db, start_date=REBALANCE_START, end_date=ANALYSIS_START,
            membership_schema=CRSP_SCHEMA.replace('_stock','_indexes'),
            stock_schema=CRSP_SCHEMA, on_unavailable='raise')
    prior=bootstrap_weights.loc[bootstrap_weights.date.le(pd.Timestamp(ANALYSIS_START)),'date']
    if prior.empty:
        raise RuntimeError('No WRDS post-rebalance snapshot exists on/before the requested analysis start')
    REPLAY_START = str(pd.Timestamp(prior.max()).date())
    print('Bootstrap/replay start:',REPLAY_START)
    print('Analysis starts:',ANALYSIS_START)
else:
    print('Using existing WRDS cache. It must have coverage.json and a prior-quarter snapshot.')
    coverage_file=PROJECT_DIR/'data/stage2_inputs/coverage.json'
    if not coverage_file.is_file():
        raise FileNotFoundError('WRDS extraction is disabled but stage2 cache is absent')
    REPLAY_START=json.loads(coverage_file.read_text())['start']

## 5. Extract four years of CRSP/WRDS data

Pulls quarterly membership/cap-weight snapshots, all relevant daily raw closes, distributions, delistings, adjustment factors and historical identifiers. **Do not proceed if WRDS reports missing coverage.** You can switch to the licensed annual CRSP release in Configuration and rerun in a fresh notebook session.

In [ ]:
from scripts.build_wrds_handoff import run as build_wrds_handoff
if RUN_WRDS_EXTRACTION:
    extraction_summary=build_wrds_handoff(
        REPLAY_START,END_DATE,REBALANCE_START,BLOOMBERG_CSV,stock_schema=CRSP_SCHEMA)
else:
    extraction_summary=json.loads((PROJECT_DIR/'data/stage2_inputs/extraction_summary.json').read_text())
print('WRDS pull summary:')
print(json.dumps(extraction_summary,indent=2))

## 6. Mandatory four-year preflight

This corrects an error in the older code: securities should **not** be required to trade on days before joining the fund or after leaving it. Check their active holdings intervals, plus the next execution session after a rebalance. Save quarterly and yearly coverage diagnostics before computing any NAV.

In [ ]:
from scripts.multiyear_preflight import audit
preflight, price_report, quarterly_report = audit(
    PROJECT_DIR,evaluation_start=ANALYSIS_START,evaluation_end=END_DATE)
print(json.dumps(preflight,indent=2))
display(quarterly_report)
display(price_report.loc[~price_report.ready].sort_values(
    ['missing_sessions','bad_price_sessions'],ascending=False).head(30))
print('Review reports: outputs/four_year_preflight/')

## 7. Corrected daily portfolio + order simulation

The scenario reuses historical CRSP raw prices, WRDS/Bloomberg corporate actions, and the same accounting ledger. It (1) applies narrow **provisional** WRDS choices where a future **CRSP-verified split** explains the dividend basis, (2) checks excluded corporate-action dates against each quarterly holding interval, and (3) checks price coverage on that interval instead of blacklisting PERMNOs across four years. Unresolved events in a holding interval still cause an explicit restricted-portfolio exclusion.

The notebook **never auto-approves** the actual `event_decisions.csv`. If the pipeline reports a blocked certification status, treat the outputs as research data only.

In [ ]:
from scripts.complete_release import generate
from traceback import format_exc
CANDIDATE = None
REPLAY_EXCEPTION = None

if RUN_REPLAY:
    try:
        CANDIDATE = generate(
            PROJECT_DIR, REPLAY_START, END_DATE,
            initial_cash=AUM,cash_buffer_bps=CASH_BUFFER_BPS,fee_bps=FEE_BPS,
            asof=str(datetime.now(timezone.utc).date()),mode='restricted',
            workbook=BLOOMBERG_XLSX, qc=BLOOMBERG_QC,
            provisional_wrds_research=PROVISIONAL_WRDS_RESEARCH)
        print('Four-year run completed. Release status:',CANDIDATE['report']['status'])
        print('Human review:',json.dumps(CANDIDATE['review_summary'],indent=2))
    except Exception as exc:
        REPLAY_EXCEPTION=f'{type(exc).__name__}: {exc}'
        (PROJECT_DIR/'outputs').mkdir(exist_ok=True)
        (PROJECT_DIR/'outputs/replay_error.txt').write_text(format_exc())
        print('Replay did NOT complete. Full portfolio/orders are NOT available.')
        print(REPLAY_EXCEPTION)
else:
    print('Replay skipped. Only extraction/preflight outputs are available.')

## 8. Verify that order book, portfolio and freeze checks are not being overstated

If a candidate exists, review its status and restricted-portfolio exclusions. Results dated before July 1, 2022 are bootstrap-only and should not be counted in the requested four-year performance evaluation. Do **not** trim accounting source files before validation.

In [ ]:
from pathlib import Path
import numpy as np
FINAL_FOLDER=Path(CANDIDATE['candidate_dir']) if CANDIDATE else None
if FINAL_FOLDER:
    report=json.loads((FINAL_FOLDER/'validation_report.json').read_text())
    nav=pd.read_csv(FINAL_FOLDER/'portfolio_nav.csv',parse_dates=['date'])
    book=pd.read_csv(FINAL_FOLDER/'orderbook.csv',parse_dates=['decision_date'])
    pos=pd.read_csv(FINAL_FOLDER/'portfolio_positions.csv',parse_dates=['date'])
    excluded=pd.read_csv(FINAL_FOLDER/'excluded_security_snapshots.csv')
    analysis_nav=nav[nav.date.ge(ANALYSIS_START)]
    print('Release:',report['status'],'| checks_passed:',report['checks_passed'])
    print('Trading sessions in evaluation period:',len(analysis_nav))
    print('Position rows in evaluation period:',int(pos.date.ge(ANALYSIS_START).sum()))
    print('Orders in evaluation period:',int(book.decision_date.ge(ANALYSIS_START).sum()))
    print('Excluded snapshot rows:',len(excluded))
    print('Validation issues:')
    display(pd.DataFrame(report.get('issues',[])).head(40))
    if len(analysis_nav)>1:
        first=analysis_nav.iloc[0].unit_nav;last=analysis_nav.iloc[-1].unit_nav
        print('Indicative evaluation-period unit NAV return:',f'{last/first-1:.3%}',
              '(restricted proxy, not benchmark verified)')
else:
    print('No replay output was produced; inspect price coverage and replay_error.txt. No dataset is frozen.')

## 8A. Verify recovered target weights per quarter

This is a **measured output from the newly rerun portfolio**, not a projection based on the old ZIP. It compares original WRDS cap-weight targets with the updated scenario's excluded weights and lists the highest remaining exclusions. If the live extraction or replay did not complete, this section will not claim any improvement.

In [ ]:
import pandas as pd
from IPython.display import display
from pathlib import Path
if CANDIDATE is not None:
    origin = pd.read_csv(PROJECT_DIR/'data/stage2_inputs/weights.csv')
    excluded = pd.read_csv(FINAL_FOLDER/'excluded_security_snapshots.csv')
    all_dates = sorted(origin.date.unique())
    loss = excluded.groupby('date').cap_weight_proxy.sum().reindex(all_dates,fill_value=0.)
    cover = pd.DataFrame({'decision_date':all_dates,
                          'excluded_target_pct':loss.values*100,
                          'retained_target_pct':(1-loss.values)*100})
    audit_folder=PROJECT_DIR/'outputs/coverage_recovery'
    audit_folder.mkdir(parents=True,exist_ok=True)
    cover.to_csv(audit_folder/'quarterly_coverage_after_live_replay.csv',index=False)
    print('EXCLUSIONS AFTER THE ACTUAL WRDS REPLAY:')
    display(cover)
    print('Range excluded:', f"{cover.excluded_target_pct.min():.2f}% – {cover.excluded_target_pct.max():.2f}%")
    print('Still excluded security-period rows:',len(excluded),'distinct PERMNO:',excluded.permno.nunique())
    print('Provisional source choices:',CANDIDATE['review_summary'].get('provisional_wrds_disagreements'))
    print('Reviewed human event approvals:',CANDIDATE['review_summary'].get('human_approved_events'))
    if cover.excluded_target_pct.max()>5:
        print('WARNING: More than 5% target weight still excluded in at least one quarter')
    if not CANDIDATE['report'].get('checks_passed',False):
        print('CERTIFICATION BLOCKED: research simulation only. Outstanding items are in review/ CSVs.')
else:
    print('No completed live replay. Do not infer corrected exclusion weights from the old ZIP.')

## 9. Prioritize the four-year corporate-action review

Ranks unresolved events by latest observed portfolio weights and event complexity. These are **review priorities only**, not automatically approved actions.

In [ ]:
from scripts.review_priority_4year import prioritize
from IPython.display import display
if CANDIDATE is not None:
    queue=prioritize(PROJECT_DIR)
    pending=queue.loc[queue.decision.fillna('PENDING').eq('PENDING')]
    print('Pending/blocked event candidates:',len(pending))
    display(pending[['review_attention','permno','event_type','ex_date','match_status',
        'last_observed_target_weight','review_key']].head(40))
    print('Saved prioritized_event_decisions_4y.csv and prioritized_scope_decisions_4y.csv')
else:
    print('Review queue cannot be ranked until a candidate has created event_decisions.csv')

## 10. Export CSVs, audit and review queues

Creates one downloadable ZIP. Licensed market-data tables should remain restricted to authorized project members. **Do not upload the workbook or WRDS raw files to public GitHub.**

Even when arithmetic passes, unresolved actions/excluded securities are explicitly marked **RESEARCH_CANDIDATE_NOT_CERTIFIED**.

In [ ]:
import zipfile
from scripts.complete_release import digest
OUT_ZIP=RUN_ROOT/f'BMLL_4Y_{ANALYSIS_START}_{END_DATE}_COVERAGE_CORRECTED_RESEARCH.zip'
folders=[
 (PROJECT_DIR/'outputs/four_year_preflight','preflight'),
 (PROJECT_DIR/'data/stage2_review','review'),
 (PROJECT_DIR/'outputs/coverage_recovery','coverage_recovery'),
]
if FINAL_FOLDER is not None:
    folders.insert(0,(FINAL_FOLDER,'candidate'))
with zipfile.ZipFile(OUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for folder,arc_prefix in folders:
        if folder.is_dir():
            for p in sorted(folder.glob('*')):
                if p.is_file() and p.suffix.lower() in ('.csv','.json','.txt'):
                    z.write(p,arcname=f'{arc_prefix}/{p.name}')
    for filename in ['extraction_summary.json','coverage.json']:
        p=PROJECT_DIR/'data/stage2_inputs'/filename
        if p.exists():z.write(p,arcname=f'preflight/{filename}')
    p=PROJECT_DIR/'outputs/replay_error.txt'
    if p.exists():z.write(p,arcname='preflight/replay_error.txt')
print('Export:',OUT_ZIP,'size MB:',round(OUT_ZIP.stat().st_size/1e6,2))
if colab_files is not None:colab_files.download(str(OUT_ZIP))
else:print('Local file:',OUT_ZIP)

## Completion criteria and what blocks a true freeze

A **four-year full-index release** additionally requires: reviewed corporate actions and scope records, no unexplained constituent exclusions or active-date price gaps, correct handling of mergers/spin-offs/delistings/fractional shares, daily index total-return benchmark, and dated BMLL identifiers for the execution handoff.

The current simulator has no L2/L3 limit-order-book reconstruction. `orderbook.csv` is the **portfolio manager's order blotter**. Its fills use an observed-close proxy, not executable closing-auction orders. Economic exceptions are surfaced as review CSVs; the code will never invent treatments to force certification.